# 02 ? Records, Collections, Metadata, and CRUD

This notebook turns the vocabulary from `01_database_fundamentals.ipynb` into a concrete data model. We will design records for document chunks, organize them into a collection, and model create, read, update, and delete operations with a tiny in-memory Python store.

The code is database-agnostic and uses only Python's standard library. It explains operations and edge cases, but it is not a vector database: it does not persist data or perform vector search. The Qdrant notebook will repeat these ideas against a real database API.

## Learning goals

By the end, you should be able to:

- design a record that links an embedding to its source chunk;
- choose stable IDs and explain their uniqueness scope;
- distinguish a record from a collection and a source document;
- choose useful metadata fields and recognize payload design pitfalls;
- explain create/insert, read, update, upsert, and delete behavior;
- reason about retries, duplicate writes, and replacing a document's chunks.

## 1. Model the data before the API

We will use this hierarchy:

**collection ? chunk records ? vector + payload**

A source document may be split into many chunks. Each chunk usually becomes one searchable record. The collection groups compatible records; each record has a unique ID, one or more vectors, and optional payload fields. A chunk is not the same thing as a document, and a collection is not necessarily one document or one tenant.

The same conceptual structure can map to different product terms and physical layouts. Keep the data model clear before learning a vendor API.

### Example source material

Suppose an internal help center has documents divided into chunks. We want search results to show the text and source, allow filtering by tenant and section, and support replacing all chunks when a document is re-ingested.

One record might look like:

```text
ID: help-101:chunk-0002
Vector: [embedding dimensions determined by the model]
Payload: text, document_id, chunk_index, section, tenant_id, source_uri
```

## 2. IDs: the address of a record

A record ID should let the application refer to the same logical chunk on a later ingestion run. Common strategies include:

- **Deterministic composite ID**, such as `document_id:chunk_index`. Readable and reconstructable, but chunk numbering can shift when chunking changes.
- **Deterministic content ID**, such as a hash of document ID plus normalized chunk text. A content change creates a new ID; old records still need cleanup.
- **Random UUID**, useful when each ingestion creates a new entity, but the application must remember the mapping to update or delete it later.

IDs must be unique within the database's relevant scope (often a collection). Stable IDs make retries safer: writing the same logical record again can replace/update it instead of creating a duplicate. Choose an ID strategy together with a source-update strategy.

In [1]:
from hashlib import sha256

def chunk_id(document_id: str, chunk_index: int) -> str:
    return f"{document_id}:chunk-{chunk_index:04d}"

def content_id(document_id: str, text: str) -> str:
    digest = sha256(text.strip().encode("utf-8")).hexdigest()[:12]
    return f"{document_id}:{digest}"

print(chunk_id("help-101", 2))
print(content_id("help-101", "Reset your password from the sign-in page."))

help-101:chunk-0002
help-101:ee0c233297f6


The first strategy is readable and stable while chunk boundaries remain stable. The second is content-sensitive. Neither strategy removes stale records automatically: if a document changes, your ingestion pipeline still has to determine which old IDs to remove.

## 3. Collections: compatibility and boundaries

A collection is a named group of records configured for a retrieval use case. Common configuration includes vector dimension and metric, with product-specific index and storage options. Some systems allow named vectors or multiple vector configurations in one collection.

Think carefully before grouping records. Records that use incompatible embedding models should not be mixed merely because their vectors have equal length. Conversely, creating one collection per document can make administration awkward. Common organizational choices include one collection per application/corpus, environment, or security boundary.

A collection's name is not a security policy by itself. Tenant isolation and permissions must be enforced by the database and application design.

## 4. Payload design

Payload (also called metadata in many APIs) contains non-vector fields associated with a record. Useful fields often fall into three groups:

| Purpose | Example fields | Why keep them? |
|---|---|---|
| Display/context | `text`, `title`, `source_uri`, `page` | Show and cite a result |
| Filtering | `tenant_id`, `language`, `category`, `published` | Restrict the candidate set |
| Maintenance/debugging | `document_id`, `chunk_index`, `ingested_at`, `content_version` | Find, replace, and diagnose records |

Payload design is a tradeoff. Duplicating full text can make retrieval convenient but increases storage and may duplicate sensitive content. Keeping only IDs reduces payload size but requires a reliable document-store lookup. Store fields that support real application operations; avoid adding everything by default.

### Metadata field types and filterability

Common payload values include strings, numbers, booleans, timestamps, and arrays. Product support and indexing behavior vary. A field can be present and readable without having an efficient filter index. Later we will create filters and learn when a database-specific payload index helps.

Use stable, consistently typed values. For example, do not store `page` as a string in some records and a number in others. Be careful with high-cardinality fields: a unique value on every record may be useful for equality lookup, but index cost and database behavior depend on the system.

## 5. Build a tiny record set

The vectors below are short toy vectors so the data model fits on screen. They are **not embeddings** and will not be used for meaningful semantic search. In a real collection, every vector must come from the selected embedding model and obey its dimension and metric requirements.

In [2]:
records = [
    {
        "id": "help-101:chunk-0000",
        "vector": [0.10, 0.20, 0.30],
        "payload": {
            "text": "To sign in, enter your work email and password.",
            "document_id": "help-101", "chunk_index": 0,
            "section": "Sign in", "tenant_id": "acme",
            "source_uri": "https://kb.example/sign-in",
        },
    },
    {
        "id": "help-101:chunk-0001",
        "vector": [0.11, 0.19, 0.31],
        "payload": {
            "text": "Reset a forgotten password from the sign-in page.",
            "document_id": "help-101", "chunk_index": 1,
            "section": "Password reset", "tenant_id": "acme",
            "source_uri": "https://kb.example/sign-in",
        },
    },
    {
        "id": "policy-205:chunk-0000",
        "vector": [0.80, 0.10, 0.10],
        "payload": {
            "text": "Contractors request access through their project sponsor.",
            "document_id": "policy-205", "chunk_index": 0,
            "section": "Access requests", "tenant_id": "acme",
            "source_uri": "https://kb.example/access-policy",
        },
    },
]

In [3]:
# A dictionary models records addressed by ID.
# A real database also persists data and builds search indexes.
store = {record["id"]: record for record in records}
len(store), list(store)

(3, ['help-101:chunk-0000', 'help-101:chunk-0001', 'policy-205:chunk-0000'])

## 6. CRUD: create, read, update, delete

CRUD is shorthand for the basic record lifecycle. Product APIs use different method names, and bulk operations, transactions, and consistency guarantees vary.

### Create / insert
Add a record with an ID that is not already present. What happens if that ID exists: fail, ignore, or replace? Know the API behavior.

### Read / retrieve
Fetch a record by ID or retrieve records matching a condition. A direct read by ID is different from nearest-vector search.

### Update
Change a record's vector, payload, or both. Clarify whether an update is a partial payload patch or complete replacement. A changed text chunk usually needs a newly generated vector; changing only the payload text leaves the stored vector representing old text.

### Delete
Remove a record by ID or condition. Deleting a source document often means deleting all its chunk records. Some APIs are eventually consistent, so a delete may not be immediately reflected in every search path.

### Create and read by ID

This in-memory analogue checks duplicate IDs and vector dimensions so the example exposes two common data-integrity rules.

In [4]:
class InMemoryCollection:
    def __init__(self, vector_size):
        self.vector_size = vector_size
        self.records = {}

    def _validate(self, record):
        if len(record["vector"]) != self.vector_size:
            raise ValueError(f"expected {self.vector_size} vector values")

    def insert(self, record):
        self._validate(record)
        if record["id"] in self.records:
            raise KeyError(f"duplicate ID: {record['id']}")
        self.records[record["id"]] = record.copy()

    def get(self, record_id):
        return self.records.get(record_id)

collection = InMemoryCollection(vector_size=3)
for item in records:
    collection.insert(item)
collection.get("help-101:chunk-0001")

{'id': 'help-101:chunk-0001',
 'vector': [0.11, 0.19, 0.31],
 'payload': {'text': 'Reset a forgotten password from the sign-in page.',
  'document_id': 'help-101',
  'chunk_index': 1,
  'section': 'Password reset',
  'tenant_id': 'acme',
  'source_uri': 'https://kb.example/sign-in'}}

This toy class stores Python objects in process memory. Restarting the notebook kernel loses them. It does not calculate nearest neighbors, create payload indexes, provide transactions, or handle concurrent writers. Those are database responsibilities we will study later.

## 7. Update, patch, replace, and upsert

These terms are easy to blur together:

- A **payload patch** changes selected fields and leaves other fields intact.
- A **replace** writes a complete new record at an existing ID. Omitted fields may disappear.
- An **upsert** commonly means insert if absent and replace/update if present, but exact semantics vary by API.

Never assume an update preserves omitted fields. Check the database API. For retryable ingestion, deterministic IDs plus well-understood upsert behavior often make repeated writes idempotent: applying the same input twice leaves one logical record.

In [5]:
def patch_payload(collection, record_id, patch):
    current = collection.get(record_id)
    if current is None:
        raise KeyError(record_id)
    updated = {**current, "payload": {**current["payload"], **patch}}
    collection.records[record_id] = updated

def upsert(collection, record):
    collection._validate(record)
    # This example's upsert has full-replacement semantics.
    collection.records[record["id"]] = record.copy()

target_id = "help-101:chunk-0000"
patch_payload(collection, target_id, {"reviewed": True})
collection.get(target_id)["payload"]

{'text': 'To sign in, enter your work email and password.',
 'document_id': 'help-101',
 'chunk_index': 0,
 'section': 'Sign in',
 'tenant_id': 'acme',
 'source_uri': 'https://kb.example/sign-in',
 'reviewed': True}

The patch kept existing text and source fields. If we instead upsert a record with the same ID but only a `reviewed` payload, this toy implementation replaces the record and drops those fields. This is why API semantics matter.

In [6]:
replacement = {
    "id": target_id,
    "vector": [0.10, 0.20, 0.30],
    "payload": {"reviewed": True},
}
upsert(collection, replacement)
collection.get(target_id)

{'id': 'help-101:chunk-0000',
 'vector': [0.1, 0.2, 0.3],
 'payload': {'reviewed': True}}

This replacement is probably not what the RAG application wants: the text and source were lost. A real ingestion operation should construct and write the complete intended record, or use a documented partial-update operation.

## 8. Delete records and replace a document

When a document is re-ingested, its new chunk boundaries may differ. Updating only the new chunk IDs can leave old chunks behind. A common simple strategy is:

1. Read/process the new document and prepare all new chunk records.
2. Write the new records using stable IDs.
3. Remove obsolete records belonging to that document.

The safest ordering and whether this can be atomic depend on database features and availability needs. For larger pipelines, use versioning, staging collections, or a manifest of record IDs so readers do not see a partially replaced document. We will revisit update pipelines in production architecture.

In [7]:
def delete(collection, record_id):
    return collection.records.pop(record_id, None) is not None

def delete_document(collection, document_id):
    to_delete = [
        record_id for record_id, record in collection.records.items()
        if record["payload"].get("document_id") == document_id
    ]
    for record_id in to_delete:
        del collection.records[record_id]
    return to_delete

delete_document(collection, "policy-205")

['policy-205:chunk-0000']

## 9. Data integrity rules worth enforcing

A production ingestion path should make its assumptions explicit. Examples:

- Each record has an ID and a vector of the expected dimension.
- Vector values are finite numbers; reject NaN and infinity.
- Required payload fields are present and have consistent types.
- `document_id` and tenant/security fields match the source system.
- Text and vector are generated from the same chunk content/version.
- Re-ingestion does not silently accumulate stale chunks.
- Retries do not create duplicate logical records.

Some checks belong in application validation, some in database constraints or schema features, and some in monitoring. Do not assume metadata is trustworthy just because it is inside the database.

## 10. Common mistakes

- **Using unstable IDs:** every retry creates another record.
- **Changing text but retaining the old vector:** payload and embedding no longer describe the same content.
- **Mixing incompatible embedding spaces:** equal dimensions can give false confidence.
- **Overloading payload:** large or sensitive fields increase storage and exposure.
- **Forgetting stale chunks:** document updates leave old passages searchable.
- **Assuming patch/upsert behavior:** omitted fields may be removed or preserved depending on the API.
- **Treating collection names as access control:** security requires actual authorization and isolation.

## Exercises

1. For an FAQ collection, propose an ID and payload schema for one chunk. Which fields support display, filters, and maintenance?
2. If you re-run ingestion with the same deterministic IDs, what must the write operation do to avoid duplicate records?
3. A source document changes from 5 chunks to 3. What could go wrong if you only upsert the 3 new chunks? Describe a cleanup strategy.
4. An update changes the payload text but not the vector. Explain the retrieval inconsistency and how to prevent it.
5. Would you store full text in the database payload or fetch it from a document store? List one benefit and cost for each design.

### Suggested answers

1. A composite ID such as `faq-42:chunk-0000`; payload might include text/title/source for display, product/language for filters, and document ID/chunk index/version for maintenance.
2. Use documented upsert/replace behavior or another idempotent write strategy keyed by that ID.
3. Two old chunks may remain. Track the document's current record IDs/version and delete obsolete IDs after preparing the replacement, with appropriate atomicity/versioning for the application.
4. The vector still represents old text. Regenerate the embedding and update vector and payload together.
5. Storing text simplifies retrieval but uses more database storage and duplicates potentially sensitive content; external lookup can reduce duplication but adds a dependency and latency/failure path.

## Summary

A record is the unit you address and maintain; a collection groups records with compatible search configuration. IDs, vectors, and payloads each serve a different purpose. CRUD details?especially upsert, patch, replacement, and delete semantics?must be understood for the chosen database. Stable IDs and a deliberate document-update strategy make ingestion safer.

**Next:** `03_storage_and_indexing.ipynb` connects this data model to persistence and the search indexes you have already studied.